# Presentation benchmark slides (unified classical + NN)

Run from the **repository root** (folder with `utils/`, `figures/cache/`) after:

1. `abr_wide_long_comparison.ipynb` — export cell writes `figures/cache/classical_benchmark_long.parquet` with **`variant`** (Brad: all-long / all-wide; Liberman: + even-long / even-wide when strain + even-SPL rows exist). Needs master comparison **`rows1`**; with strain also **`rows2`**, **`rows6`**.
2. `abr_nn_stage2.ipynb` — export cell writes `figures/cache/nn_metrics_all.parquet`.

Outputs **`figures/presentation/benchmark_r2`** and **`benchmark_rmse`** (**PNG+SVG**, 300 dpi PNG): **2×2** strip plots (train **B** vs matched **A/C**), classical points by **variant** + NN squares; digests **`=== S3 benchmark_r2 digest ===`** / **`=== S3 benchmark_rmse digest ===`**; **Stage 1** optional **RF bar chart** + **LR/RF ROC + calibration** (from `stage1_wide_noise_lr_rf_eval.parquet`).

**Typography:** **`apply_slide_rcparams()`** — talk context + slide font sizes.

**Unified plots:** strip/dot chart (not bars). **Stage 1:** optional grouped bars from JSON (RF-centric summary) plus **ROC + calibration** **PNG+SVG** from the animal-level eval Parquet when present.

**Phase 2:** multi-seed `figures/cache/benchmark_metrics_by_seed.parquet` merged into `benchmark_df` for SEM columns (strip plots do not draw SEM bars).

In [1]:
%matplotlib inline

from pathlib import Path

import pandas as pd

from utils.benchmark_metrics import (
    BENCHMARK_MERGED_PARQUET,
    STAGE1_WIDE_LR_RF_EVAL_PARQUET,
    apply_slide_rcparams,
    benchmark_merged_has_classical_variants,
    load_benchmark_from_cache,
    plot_stage1_bars,
    plot_stage1_lr_rf_roc_and_calibration,
    plot_unified_benchmark,
    summarize_best_models,
)

apply_slide_rcparams()  # sns.set_context("talk") + slide font sizes (see utils/benchmark_metrics.py)

OUT_DIR = Path("figures/presentation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Prefer merged cache only if it includes **variant** (skip stale pre-variant files).
USE_MERGED_CACHE = True
if (
    USE_MERGED_CACHE
    and BENCHMARK_MERGED_PARQUET.exists()
    and benchmark_merged_has_classical_variants(BENCHMARK_MERGED_PARQUET)
):
    benchmark_df = pd.read_parquet(BENCHMARK_MERGED_PARQUET)
else:
    benchmark_df = load_benchmark_from_cache(save_merged=BENCHMARK_MERGED_PARQUET)

benchmark_df.head()

,test_set,scenario,model,variant,R2,RMSE,source
0,Brad,A,LR baseline,all-long,0.179224,4.300470,classical
1,Brad,A,LR baseline,all-wide,0.374009,3.755672,classical
2,Brad,B,LR baseline,all-long,0.179224,4.300470,classical
3,Brad,B,LR baseline,all-wide,0.374009,3.755672,classical
4,Brad,C,LR baseline,all-long,0.179224,4.300470,classical


In [2]:
plot_unified_benchmark(
    benchmark_df,
    metric="R2",
    out_path=OUT_DIR / "benchmark_r2.png",
)
plot_unified_benchmark(
    benchmark_df,
    metric="RMSE",
    out_path=OUT_DIR / "benchmark_rmse.png",
)

best_by_r2 = summarize_best_models(benchmark_df)
best_by_r2

/var/folders/77/zyk5tmbd30d8p7jw_rxvl4cc0000gn/T/ipykernel_42479/741770696.py:1: UserWarning: omit_liberman_scenario_a_r2 is ignored for the 2×2 stripplot layout.
  plot_unified_benchmark(
/var/folders/77/zyk5tmbd30d8p7jw_rxvl4cc0000gn/T/ipykernel_42479/741770696.py:6: UserWarning: omit_liberman_scenario_a_r2 is ignored for the 2×2 stripplot layout.
  plot_unified_benchmark(


,test_set,model,best_R2,RMSE_at_best_R2,scenario_at_best,variant_at_best
0,Brad,LR baseline,0.374009,3.755672,A,all-wide
1,Brad,LR full,0.535763,3.234250,A,all-long
2,Brad,RF,0.555613,3.164349,A,all-long
3,Brad,XGB,0.571977,3.105539,A,all-long
4,Brad,MLP,0.555100,3.166100,B,NaN
5,Brad,CNN (Wave I),0.579000,3.080100,B,NaN
6,Brad,CNN (full wave),0.569500,3.114400,B,NaN
7,Liberman,LR baseline,0.228920,3.090075,A,all-wide
8,Liberman,LR full,0.470201,2.561385,A,even-wide
9,Liberman,RF,0.641076,2.108242,C,even-wide


### Stage 1 (optional)

- **Bars (S4):** `figures/cache/stage1_wide_rf_metrics.json` from `abr_stage1_wide.ipynb` (selected RF or LR) → **`stage1_wide_rf.png`** + **`.svg`**; digest **`=== S4 stage1_wide_rf bars digest ===`**.
- **ROC + calibration:** `figures/cache/stage1_wide_noise_lr_rf_eval.parquet` (LR + RF, animal-level) from the same notebook export cell.

In [3]:
from pathlib import Path
from utils.stage1_wide_report import load_stage1_metrics
from utils.benchmark_metrics import plot_stage1_bars, plot_stage1_lr_rf_roc_and_calibration

stage1_metrics = load_stage1_metrics()
if stage1_metrics:
    plot_stage1_bars(stage1_metrics, OUT_DIR / 'stage1_wide_rf.png')
else:
    print('Skip Stage 1 figure — run abr_stage1_wide.ipynb first')


Skip Stage 1 figure — run NN notebook Stage 1 cell first: figures/cache/stage1_wide_rf_metrics.json


### Multi-seed SEM (later)

Phase 2 multi-seed SEM (test R² / RMSE): repeat full fit + evaluation with fixed splits
and seeds 1..n; save long Parquet:

    seed, test_set, scenario, model, R2, RMSE

Merge with `attach_seed_sem()` or `load_benchmark_from_cache()` when
`figures/cache/benchmark_metrics_by_seed.parquet` exists.